This is the final inference notebook. You can read our solution [here](https://www.kaggle.com/competitions/equity-post-HCT-survival-predictions/discussion/566522).

In [ ]:
!pip install -q --no-deps /kaggle/input/pytabkit-whl/pytabkit-1.2.1-py3-none-any.whl
!pip install -q /kaggle/input/cibmtr-dataset/packages/scikit_learn-1.5.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
!pip install -q /kaggle/input/cibmtr-dataset/packages/xgboost-2.1.3-py3-none-manylinux_2_28_x86_64.whl
!pip install -q --no-deps /kaggle/input/download-lightning-and-pytorch-tabular/pytorch_tabular-1.1.1-py2.py3-none-any.whl
!pip install -q --no-deps /kaggle/input/download-lightning-and-pytorch-tabular/pytorch_tabnet-4.1.0-py3-none-any.whl

## First Pipeline

#### 1. Stage - Classification for efs prediction
* HistGBM
* CatBoost
  
#### 2. Stage - Regression with efs=1 on inference
* HistGBM

For a more detailed version: [Güneş's github repo](https://github.com/gunesevitan/cibmtr-equity-in-post-hct-survival-predictions)

In [ ]:
import warnings
from pathlib import Path
import pickle
import yaml
from tqdm import tqdm
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
import catboost as cb

In [ ]:
warnings.filterwarnings('ignore')

competition_dataset_directory = Path('/kaggle/input/equity-post-HCT-survival-predictions')
external_dataset_directory = Path('/kaggle/input/cibmtr-dataset')

pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_columns', 1000)

In [ ]:
df = pd.read_csv(competition_dataset_directory / 'test.csv')
print(f'Dataset Shape: {df.shape}')

In [ ]:
def one_hot_encode_categorical_columns(df, categorical_columns, transformer_directory, load_transformers=False):

    """
    One-hot encode given categorical columns and concatenate them to given dataframe

    Parameters
    ----------
    df: pandas.DataFrame
        Dataframe with categorical columns

    categorical_columns: list
        List of categorical columns

    transformer_directory: str or pathlib.Path
        Path of the serialized transformers

    load_transformers: bool
        Whether to load transformers from the given transformer directory or not

    Returns
    -------
    df: pandas.DataFrame
        Dataframe with encoded categorical columns
    """

    Path(transformer_directory).mkdir(parents=True, exist_ok=True)

    for column in categorical_columns:

        if load_transformers:

            with open(transformer_directory / f'{column}_one_hot_encoder.pickle', mode='rb') as f:
                encoder = pickle.load(f)

            encoded = encoder.transform(df[column].values.reshape(-1, 1))
            encoded = pd.DataFrame(encoded, columns=encoder.get_feature_names_out([column]), index=df.index)
            df = pd.concat((df, encoded), axis=1, ignore_index=False)

        else:

            encoder = OneHotEncoder(
                categories='auto',
                drop=None,
                sparse_output=False,
                dtype=np.uint8,
                handle_unknown='ignore',
                min_frequency=128
            )
            encoded = encoder.fit_transform(df[column].values.reshape(-1, 1))
            encoded = pd.DataFrame(encoded, columns=encoder.get_feature_names_out([column]), index=df.index)
            df = pd.concat((df, encoded), axis=1, ignore_index=False)

            with open(transformer_directory / f'{column}_one_hot_encoder.pickle', mode='wb') as f:
                pickle.dump(encoder, f)

    return df


def ordinal_encode_categorical_columns(df, categorical_columns, transformer_directory, load_transformers=False):

    """
    Ordinal encode given categorical columns and concatenate them to given dataframe

    Parameters
    ----------
    df: pandas.DataFrame
        Dataframe with categorical columns

    categorical_columns: list
        List of categorical columns

    transformer_directory: str or pathlib.Path
        Path of the serialized transformers

    load_transformers: bool
        Whether to load transformers from the given transformer directory or not

    Returns
    -------
    df: pandas.DataFrame
        Dataframe with encoded categorical columns
    """

    Path(transformer_directory).mkdir(parents=True, exist_ok=True)

    for column in categorical_columns:

        column_dtype = df[column].dtype
        fill_value = 'missing' if column_dtype == object else -1

        if load_transformers:

            with open(transformer_directory / f'{column}_ordinal_encoder.pickle', mode='rb') as f:
                encoder = pickle.load(f)

            df[f'{column}_encoded'] = encoder.transform(df[column].fillna(fill_value).values.reshape(-1, 1))

        else:

            encoder = OrdinalEncoder(
                categories='auto',
                dtype=np.int32,
                handle_unknown='use_encoded_value',
                unknown_value=-1,
                encoded_missing_value=np.nan,
            )
            df[f'{column}_encoded'] = encoder.fit_transform(df[column].fillna(fill_value).values.reshape(-1, 1))

            with open(transformer_directory / f'{column}_ordinal_encoder.pickle', mode='wb') as f:
                pickle.dump(encoder, f)

    return df


def normalize_continuous_columns(df, continuous_columns, transformer_directory, load_transformers=False):

    """
    Normalize continuous columns and concatenate them to given dataframe

    Parameters
    ----------
    df: pandas.DataFrame
        Dataframe with continuous columns

    continuous_columns: list
        List of continuous columns

    transformer_directory: str or pathlib.Path
        Path of the serialized transformers

    load_transformers: bool
        Whether to load transformers from the given transformer directory or not

    Returns
    -------
    df: pandas.DataFrame
        Dataframe with encoded continuous columns
    """

    Path(transformer_directory).mkdir(parents=True, exist_ok=True)

    if load_transformers:

        with open(transformer_directory / 'standard_scaler.pickle', mode='rb') as f:
            normalizer = pickle.load(f)

        normalized_column_names = [f'{column}_normalized' for column in continuous_columns]
        df[normalized_column_names] = normalizer.transform(df[continuous_columns].fillna(df[continuous_columns].median()).values)

    else:

        normalizer = StandardScaler()
        normalizer.fit(df[continuous_columns].values)
        normalized_column_names = [f'{column}_normalized' for column in continuous_columns]
        df[normalized_column_names] = normalizer.transform(df[continuous_columns].fillna(df[continuous_columns].median()).values)

        with open(transformer_directory / 'standard_scaler.pickle', mode='wb') as f:
            pickle.dump(normalizer, f)

    return df


In [ ]:
def cast_categorical_columns(df, categorical_columns, dtype):

    """
    Cast given categorical columns to category or str

    Parameters
    ----------
    df: pandas.DataFrame
        Dataframe with categorical columns

    categorical_columns: list
        Array of categorical column names

    dtype: str ('category' or str)
        Data type of the categorical column

    Returns
    -------
    df: pandas.DataFrame
        Dataframe with casted categorical columns
    """

    for column in categorical_columns:
        df[f'{column}_{dtype.__name__ if dtype == str else dtype}'] = df[column].astype(dtype)

    return df


In [ ]:
categorical_columns = [
    'dri_score', 'psych_disturb', 'cyto_score', 'diabetes', 'hla_match_c_high', 'hla_high_res_8', 'tbi_status',
    'arrhythmia', 'hla_low_res_6', 'graft_type', 'vent_hist', 'renal_issue', 'pulm_severe', 'prim_disease_hct',
    'hla_high_res_6', 'cmv_status', 'hla_high_res_10', 'hla_match_dqb1_high', 'tce_imm_match', 'hla_nmdp_6',
    'hla_match_c_low', 'rituximab', 'hla_match_drb1_low', 'hla_match_dqb1_low', 'prod_type', 'cyto_score_detail',
    'conditioning_intensity', 'ethnicity', 'year_hct', 'obesity', 'mrd_hct', 'in_vivo_tcd', 'tce_match',
    'hla_match_a_high', 'hepatic_severe', 'prior_tumor', 'hla_match_b_low', 'peptic_ulcer', 'hla_match_a_low',
    'gvhd_proph', 'rheum_issue', 'sex_match', 'hla_match_b_high', 'race_group', 'comorbidity_score', 'karnofsky_score',
    'hepatic_mild', 'tce_div_match', 'donor_related', 'melphalan_dose', 'hla_low_res_8', 'cardiac', 'hla_match_drb1_high',
    'pulm_moderate', 'hla_low_res_10'
]

continuous_columns = [
    'donor_age', 'age_at_hct'
]

transformer_directory = external_dataset_directory / 'transformers'

In [ ]:
df = df.fillna(np.nan)

# One-hot encode categorical columns for linear models
df = one_hot_encode_categorical_columns(
    df=df,
    categorical_columns=categorical_columns,
    transformer_directory=transformer_directory,
    load_transformers=True
)

# Ordinal encode categorical columns for tree models
df = ordinal_encode_categorical_columns(
    df=df,
    categorical_columns=categorical_columns,
    transformer_directory=transformer_directory,
    load_transformers=True
)

# Normalize continuous columns for linear models
df = normalize_continuous_columns(
    df=df,
    continuous_columns=continuous_columns,
    transformer_directory=transformer_directory,
    load_transformers=True
)

# Cast categorical columns to category for LightGBM and XGBoost
df = cast_categorical_columns(
    df=df,
    categorical_columns=categorical_columns + continuous_columns,
    dtype='category'
)

# Cast categorical columns to str for CatBoost
df = cast_categorical_columns(
    df=df,
    categorical_columns=categorical_columns + continuous_columns,
    dtype=str
)

print(f'Transformed Test Set Shape: {df.shape}')

In [ ]:
def load_sklearn_models(model_directory):

    config_path = model_directory / 'config.yaml'
    config = yaml.load(open(config_path), Loader=yaml.FullLoader)

    models = {}

    for model_path in tqdm(sorted(list(model_directory.glob('model*')))):
        model_path = str(model_path)
        with open(model_path, mode='rb') as f:
            model = pickle.load(f)
        model_file_name = model_path.split('/')[-1].split('.')[0]
        models[model_file_name] = model
        print(f'Loaded {model.__class__.__name__} from {model_path}')

    return config, models


def sklearn_predict(df, model_name, config, models):

    prediction_column = f'{model_name}_prediction'
    df[prediction_column] = 0.

    for model_file_name, model in tqdm(models.items()):

        if config['training']['task'] == 'classification':
            model_predictions = model.predict_proba(df[config['training']['features']])[:, 1]
        else:
            model_predictions = model.predict(df[config['training']['features']])

        print(f'{model.__class__.__name__} Model {model_file_name} Predictions - Mean: {np.mean(model_predictions):.4f} Std: {np.std(model_predictions):.4f} Min: {np.min(model_predictions):.4f} Max: {np.max(model_predictions):.4f}')

        df[prediction_column] += model_predictions / len(models)

    return df


In [ ]:
hist_gbm_efs_config, hist_gbm_efs_models = load_sklearn_models(model_directory=external_dataset_directory / 'hist_gradient_boosting_classifier_efs')

hist_gbm_log_efs_time_config, hist_gbm_log_efs_time_models = load_sklearn_models(model_directory=external_dataset_directory / 'hist_gradient_boosting_regressor_log_efs_time')

In [ ]:
def load_cb_model(model_directory, task):

    config_path = model_directory / 'config.yaml'
    config = yaml.load(open(config_path), Loader=yaml.FullLoader)

    models = {}

    for model_path in tqdm(sorted(list(model_directory.glob('model*')))):
        model_path = str(model_path)
        if task == 'regression':
            model = cb.CatBoostRegressor()
        elif task == 'classification':
            model = cb.CatBoostClassifier()
        model.load_model(model_path)
        model_file_name = model_path.split('/')[-1].split('.')[0]
        models[model_file_name] = model
        print(f'Loaded CatBoost Model from {model_path}')

    return config, models


def cb_predict(df, model_name, config, models):

    prediction_column = f'{model_name}_prediction'
    df[prediction_column] = 0.

    for model_file_name, model in tqdm(models.items()):
        
        if isinstance(model, cb.CatBoostClassifier):
            model_predictions = model.predict_proba(df[config['training']['features']])[:, 1]
        else:
            model_predictions = model.predict(df[config['training']['features']])

        print(f'CatBoost Model {model_file_name} Predictions - Mean: {np.mean(model_predictions):.4f} Std: {np.std(model_predictions):.4f} Min: {np.min(model_predictions):.4f} Max: {np.max(model_predictions):.4f}')

        df[prediction_column] += model_predictions / len(models)

    return df


In [ ]:
cb_efs_binary_classifier_config, cb_efs_binary_classifier_models = load_cb_model(external_dataset_directory / 'catboost_efs_binary_classifier', task='classification')

In [ ]:
df = sklearn_predict(
    df=df,
    model_name='hist_gbm_efs',
    config=hist_gbm_efs_config,
    models=hist_gbm_efs_models
)

df = cb_predict(
    df=df,
    model_name='cb_efs',
    config=cb_efs_binary_classifier_config,
    models=cb_efs_binary_classifier_models
)

In [ ]:
df = sklearn_predict(
    df=df,
    model_name='hist_gbm_log_efs_time',
    config=hist_gbm_log_efs_time_config,
    models=hist_gbm_log_efs_time_models
)

In [ ]:
predictions_to_save = [
    'hist_gbm_efs_prediction', 'cb_efs_prediction',
    'hist_gbm_log_efs_time_prediction'
]
df_first_pipe_predictions = df[predictions_to_save].rename(columns={column: f'first_pipe_{column}' for column in predictions_to_save})
df_first_pipe_predictions

In [ ]:
df_first_pipe_predictions

---
---

## Second Pipeline

#### 1. Stage - Classification for efs prediction
* [RealMLP](https://pytabkit.readthedocs.io/en/latest/index.html)
* Catboost

Mean ensemble of two estimators.

#### 2. Stage - Regression with efs=1 on inference
* XGBoost

#### 3. Stage - Custom NN

Additionally, we train a neural network which approximates the competition metric and directly predicts the risk scores. It uses 2.stage regression predictions from all pipelines and optimizes the approximate competition metric and auxiliary binary classification loss.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.metrics import mean_squared_error,mean_absolute_error,roc_auc_score

import xgboost as xgb
from catboost import CatBoostRegressor, CatBoostClassifier
from pytabkit import RealMLP_TD_Classifier

import joblib
import cloudpickle
from typing import List
from pytorch_tabular.models.common.layers import ODST

test = pd.read_csv("/kaggle/input/equity-post-HCT-survival-predictions/test.csv")

In [ ]:
RMV = ["ID","efs","efs_time"]
FEATURES = [c for c in test.columns if not c in RMV]
print(f"There are {len(FEATURES)} FEATURES: {FEATURES}")

In [ ]:
CATS = [c for c in FEATURES if not c in ['age_at_hct', 'donor_age']]
print(f"There are {len(CATS)} CAT FEATURES: {CATS}") 

In [ ]:
models_path = "/kaggle/input/pytabkit-2stage/2stage-6940-v2"

X_test_clf = test[FEATURES]
X_test_reg = test[FEATURES]

X_test_clf = X_test_clf.fillna(-99)
X_test_reg = X_test_reg.fillna(-99)

In [ ]:
for c in CATS:
    X_test_clf[c] = X_test_clf[c].astype("str").fillna("NAN").astype("category")
    X_test_reg[c] = X_test_reg[c].astype("str").fillna("NAN").astype("category")

In [ ]:
X_test_clf

In [ ]:
test_preds_clf_mlp = []
test_preds_clf_cat = []
test_preds_reg1 = []

X_test_reg["efs"] = 1

for fold in range(0,7):
    ###Classifier###
    #REALMLP
    model_clf_mlp = cloudpickle.load(open(f"{models_path}/classifier_{fold}.pkl",'rb'))
    test_preds_clf_mlp.append(model_clf_mlp.predict_proba(X_test_clf)[:, 1])
    #CATBOOST
    model_clf_cat = joblib.load(f"{models_path}/classifier_catboost_{fold}.pkl")
    test_preds_clf_cat.append(model_clf_cat.predict_proba(X_test_clf)[:, 1])

    ###Regressor###
    model_reg = joblib.load(f"{models_path}/regressor_{fold}.pkl")
    test_preds_reg1.append(model_reg.predict(X_test_reg))


In [ ]:
test['second_pipe_pred_clf_mlp'] = np.stack(test_preds_clf_mlp).mean(axis=0)
test['second_pipe_pred_clf_cat'] = np.stack(test_preds_clf_cat).mean(axis=0)
test["second_pipe_pred_reg1"] = np.stack(test_preds_reg1).mean(axis=0) #efs=1

test[["second_pipe_pred_clf_mlp", "second_pipe_pred_clf_cat", "second_pipe_pred_reg1"]]

---
---

In [ ]:
df = pd.concat([df_first_pipe_predictions, test[["ID", "second_pipe_pred_clf_mlp", "second_pipe_pred_clf_cat", "second_pipe_pred_reg1"]]], axis=1)
df

In [ ]:
clf_ens = (0.3*df['second_pipe_pred_clf_mlp'] + 0.2*df['second_pipe_pred_clf_cat'] +
           0.3*df['first_pipe_hist_gbm_efs_prediction'] + 0.2*df['first_pipe_cb_efs_prediction']).values

#### 3. Stage - Custom NN

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))
    
    
reg_ens1 = (0.7*df["second_pipe_pred_reg1"] + 0.3*df["first_pipe_hist_gbm_log_efs_time_prediction"]).values

final_pred = clf_ens * sigmoid(-1.1*reg_ens1)

In [ ]:
df = pd.read_csv("/kaggle/input/equity-post-HCT-survival-predictions/test.csv")
df.shape

In [ ]:
numerics = ['age_at_hct', 'donor_age']

df[[f"{col}_cat" for col in numerics]] = df[numerics] // 8

features = [col for col in df.columns if col not in {"ID", "efs", "efs_time"}]
len(features)

In [ ]:

categoricals = [col for col in features if col not in numerics]

df[categoricals] = df[categoricals].astype(str).fillna("__null__")

In [ ]:
import numpy as np
import joblib
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from collections import Counter

class LabelEncoderMinFreq(BaseEstimator, TransformerMixin):
    def __init__(self, min_frequency=1):
        self.min_frequency = min_frequency
        self.category_maps = {}
        self.other_label = "__oTHeR__"
    
    def fit(self, X, y=None):
        if not isinstance(X, (pd.DataFrame, np.ndarray, list)):
            raise ValueError("Input should be a pandas DataFrame, list, or numpy array")
        
        X = pd.DataFrame(X) if not isinstance(X, pd.DataFrame) else X
        
        for col in X.columns:
            counts = Counter(X[col])
            self.category_maps[col] = dict()
            for i, (cat, freq) in enumerate(counts.items()):
                if freq >= self.min_frequency:
                    self.category_maps[col][cat] = len(self.category_maps[col])
            self.category_maps[col][self.other_label] = len(self.category_maps[col])
        
        return self
    
    def transform(self, X):
        if not isinstance(X, (pd.DataFrame, np.ndarray, list)):
            raise ValueError("Input should be a pandas DataFrame, list, or numpy array")
        
        X = pd.DataFrame(X) if not isinstance(X, pd.DataFrame) else X
        
        return X.apply(lambda col: col.map(self.category_maps[col.name]).fillna(self.category_maps[col.name][self.other_label])).to_numpy().astype(int)
    
    def fit_transform(self, X, y=None):
        return self.fit(X, y).transform(X)
    
    def inverse_transform(self, X):
        X = pd.DataFrame(X)
        
        return X.apply(lambda col: col.map({v: k for k, v in self.category_maps[col.name].items()}).fillna(self.other_label)).to_numpy()
    
    def save(self, filepath):
        joblib.dump(self, filepath)
    
    @staticmethod
    def load(filepath):
        return joblib.load(filepath)

In [ ]:
df["reg0_pred"] = 0
df["reg1_pred"] = reg_ens1

df[["reg0_pred", "reg1_pred"]]

In [ ]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torch
import torch.nn as nn
import os
from tqdm import tqdm
import sys


NW = 8
BS = 256
    

class CIBMTRDataset(Dataset):
    def __init__(self, df, cats, is_train=False):
        self.df = df.reset_index(drop=True)
        self.is_train = is_train
        self.cats = cats

    def __len__(self):
        return self.df.shape[0]

    def __getitem__(self, index):
        row = self.df.iloc[index]
        
        reg = row[["reg0_pred", "reg1_pred"]]
        
        return torch.LongTensor(self.cats[index]), torch.FloatTensor(reg)

In [ ]:
from typing import List
from pytorch_tabular.models.common.layers import ODST


class CatEmbeddings(nn.Module):
    """
    Embedding module for the categorical dataframe.
    """
    def __init__(
        self,
        projection_dim: int,
        categorical_cardinality: List[int],
        embedding_dim: int
    ):
        """
        projection_dim: The dimension of the final output after projecting the concatenated embeddings into a lower-dimensional space.
        categorical_cardinality: A list where each element represents the number of unique categories (cardinality) in each categorical feature.
        embedding_dim: The size of the embedding space for each categorical feature.
        self.embeddings: list of embedding layers for each categorical feature.
        self.projection: sequential neural network that goes from the embedding to the output projection dimension with GELU activation.
        """
        super(CatEmbeddings, self).__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(cardinality, embedding_dim)
            for cardinality in categorical_cardinality
        ])
        self.projection = nn.Sequential(
            nn.Linear(embedding_dim * len(categorical_cardinality), projection_dim),
            nn.GELU(),
            nn.Linear(projection_dim, projection_dim)
        )

    def forward(self, x_cat):
        """
        Apply the projection on concatened embeddings that contains all categorical features.
        """
        x_cat = [embedding(x_cat[:, i]) for i, embedding in enumerate(self.embeddings)]
        x_cat = torch.cat(x_cat, dim=1)
        return self.projection(x_cat)
    

    
class CIBMTRModel(nn.Module):
    def __init__(self, le, model_size=56, emb_dim=16, dropout=0.05):
        super(CIBMTRModel, self).__init__()
        self.mlp = nn.Sequential(
            ODST(model_size*2, model_size),
            nn.BatchNorm1d(model_size),
            nn.Dropout(dropout)
        )
        
        self.dropout = nn.Dropout(dropout)
        
        self.embeddings = CatEmbeddings(projection_dim=model_size*2, 
                                        categorical_cardinality=[len(le.category_maps[col]) for col in categoricals], 
                                        embedding_dim=emb_dim)
        
        self.cls_head = nn.Linear(model_size, 1)
        self.risk_head = nn.Sequential(nn.Linear(1, 16), nn.LeakyReLU(),
                                       nn.Linear(16, 16), nn.LeakyReLU(),
                                       nn.Linear(16, 1))
        
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)
        

    def forward(self, c, reg):
        x = self.embeddings(c)
        x = self.dropout(x)
        x = self.mlp(x)
        
        #x0 = self.layer2(torch.cat([x_init, torch.zeros(x_init.shape[0], 1).to(x_init.device)], axis=1))
        #x1 = self.layer2(torch.cat([x_init, torch.ones(x_init.shape[0], 1).to(x_init.device)], axis=1))
        
        #reg = torch.cat([x0, x1], axis=1)
        
        cls = self.cls_head(x)
        
        risk = self.risk_head(cls.sigmoid()*(-reg[:, 1:]).sigmoid())
        
        return risk, reg, cls


    def save(self, path):
        torch.save(self.state_dict(), path)

    def load(self, path):
        self.load_state_dict(torch.load(path, map_location='cpu'))

In [ ]:
def infer(model, val_loader):    
    model.eval()

    tbar = tqdm(val_loader, file=sys.stdout)

    #loss_list = []
    pred_list = []

    with torch.no_grad():
        for idx, (c, reg) in enumerate(tbar):
            c = c.cuda()
            
            preds = model(c, reg.cuda())[0]
            pred_list.append(preds.detach().cpu().numpy().ravel())
    
    return np.concatenate(pred_list)


y_test = np.zeros(df.shape[0])

N_FOLDS = 4
N_ITERS = 2
VERSION = "v32"

for fold in range(N_FOLDS):
    
    le = LabelEncoderMinFreq.load(f"/kaggle/input/cibmtr-{VERSION}/cibmtr_{VERSION}/le_{VERSION}_{fold}.joblib")
    model = CIBMTRModel(le).eval()
    model = model.cuda()

    ds = CIBMTRDataset(df, le.transform(df[categoricals]))
    loader = DataLoader(ds, batch_size=BS, shuffle=False, num_workers=NW,
                                 pin_memory=False, drop_last=False) 

    for it in range(N_ITERS):
        print("Iteration", it)

        model.load(f"/kaggle/input/cibmtr-{VERSION}/cibmtr_{VERSION}/{VERSION}_{fold}_{it}.pth")

        pred = infer(model, loader)
        #print(pred)
        y_test += pred / (N_ITERS * N_FOLDS)

### Weighted Ensemble of 2 Pipelines

In [ ]:
df["ens_pred"] = final_pred
df["nn_pred"] = y_test

df["prediction"] = 0.8*df["ens_pred"].rank(pct=True) + 0.2*df["nn_pred"].rank(pct=True)
df["prediction"] = df["prediction"].rank(pct=True, method="first")

df[["ID", "prediction", "ens_pred", 'nn_pred']]

In [ ]:
df.to_csv("submission.csv", index=False, columns=["ID", "prediction"])